In [ ]:
############################################################
# Re-ranking Recommendation (Hybrid)
############################################################

import pandas as pd
import numpy as np
import os
import joblib
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
from surprise import SVD, Reader, Dataset

# ---------------------------------------------------------
# 1️ 경로 설정 및 파일 로드
# ---------------------------------------------------------
BASE_DIR = '../'
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# 필수 파일 로드
meta_path = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')
tfidf_path = os.path.join(OUTPUT_DIR, 'tfidf_embedding_matrix.npz')
svd_path = os.path.join(OUTPUT_DIR, 'svd_collaborative_instrument2music.joblib')
trend_path = os.path.join(OUTPUT_DIR, 'popularity_model_top_items.csv')
reviews_path = os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv')

meta_df = pd.read_pickle(meta_path)
tfidf_matrix = load_npz(tfidf_path)
svd_model = joblib.load(svd_path)
trend_df = pd.read_csv(trend_path)
reviews_df = pd.read_csv(reviews_path)

# ---------------------------------------------------------
# 2️ TF-IDF 행렬과 메타데이터 인덱스 동기화
# ---------------------------------------------------------
meta_df = meta_df.reset_index(drop=True)
if len(meta_df) != tfidf_matrix.shape[0]:
    print(f"Warning: meta_df({len(meta_df)}) ≠ tfidf_matrix({tfidf_matrix.shape[0]})")
    # 자동 동기화 수행 (작을 쪽 기준으로 맞춤)
    min_len = min(len(meta_df), tfidf_matrix.shape[0])
    meta_df = meta_df.iloc[:min_len].reset_index(drop=True)
    tfidf_matrix = tfidf_matrix[:min_len, :]
    print(f"동기화 완료: meta_df({len(meta_df)}), tfidf_matrix({tfidf_matrix.shape[0]})")

asin_to_index = pd.Series(meta_df.index, index=meta_df['asin'])

# ---------------------------------------------------------
# 3️ 콘텐츠 기반 유사도 함수 (CB)
# ---------------------------------------------------------
def get_cb_score(target_asin):
    """특정 아이템과 유사한 아이템의 코사인 유사도 반환"""
    if target_asin not in asin_to_index:
        print(f"[CB] '{target_asin}' asin이 메타데이터에 없습니다.")
        return pd.Series(dtype=float)

    idx = asin_to_index[target_asin]

    # 인덱스 범위 검사
    if idx >= tfidf_matrix.shape[0]:
        print(f"[CB] 인덱스 {idx}가 TF-IDF 행렬 범위를 벗어났습니다.")
        return pd.Series(dtype=float)

    # 유사도 계산
    target_vec = tfidf_matrix[idx]
    sims = cosine_similarity(target_vec, tfidf_matrix).flatten()

    return pd.Series(sims, index=meta_df['asin'])

# ---------------------------------------------------------
# 4️ 협업 필터링 예측 함수 (CF)
# ---------------------------------------------------------
def get_cf_score(user_id):
    """특정 사용자에 대한 SVD 예측 평점 반환"""
    all_items = meta_df['asin'].unique()
    cf_scores = []
    for item in all_items:
        try:
            est = svd_model.predict(user_id, item).est
        except Exception:
            est = 0.0
        cf_scores.append(est)
    cf_scores = pd.Series(cf_scores, index=all_items)
    return cf_scores

# ---------------------------------------------------------
# 5️ 트렌드 점수 정규화
# ---------------------------------------------------------
trend_scores = trend_df[['asin', 'score']].set_index('asin')['score']
trend_scores = (trend_scores - trend_scores.min()) / (trend_scores.max() - trend_scores.min())
trend_scores = trend_scores.reindex(meta_df['asin']).fillna(0)

# ---------------------------------------------------------
# 6  하이브리드 리랭킹 함수
# ---------------------------------------------------------
def hybrid_recommend(user_id, liked_asin, top_n=10, alpha=0.5, beta=0.3, gamma=0.2):
    """
    hybrid 추천: CF, CB, Trend 기반 점수 결합 후 Top-N 반환
    """
    # ① CF 점수 (사용자 개인화)
    cf = get_cf_score(user_id)
    cf = (cf - cf.min()) / (cf.max() - cf.min() + 1e-9)  # 0-1 정규화

    # ② CB 점수 (아이템 유사도)
    cb = get_cb_score(liked_asin)
    if cb.empty:
        cb = pd.Series(0, index=meta_df['asin'])
    cb = (cb - cb.min()) / (cb.max() - cb.min() + 1e-9)

    # ③ Trend 점수 (전역 인기)
    trend = trend_scores.copy()

    # ④ 세 모델 점수 결합 (weighted sum)
    hybrid = alpha * cf + beta * cb + gamma * trend
    hybrid = hybrid.drop(liked_asin, errors='ignore')

    # ⑤ Top-N 추출
    top_items = hybrid.sort_values(ascending=False).head(top_n)

    # ⑥ 메타데이터 병합
    result = meta_df.set_index('asin').loc[top_items.index, ['title_str', 'domain']].copy()
    result['final_score'] = top_items.values

    return result.reset_index(names='asin')

# ---------------------------------------------------------
# 7️ 예시 실행
# ---------------------------------------------------------
USER_ID = reviews_df['reviewerID'].iloc[0]
LIKED_ASIN = reviews_df['asin'].iloc[0]

print(f"\n실행 예시: 사용자 {USER_ID}, 기준 아이템 {LIKED_ASIN}\n")

recommendations = hybrid_recommend(USER_ID, LIKED_ASIN, top_n=10)
print("\n🎯 최종 하이브리드 추천 결과 (리랭킹):")
print(recommendations.to_markdown(index=False))



실행 예시: 사용자 AGJSTXSEWFVQWVQINO6QKMNBUUNA, 기준 아이템 B0037JT95I


🎯 최종 하이브리드 추천 결과 (리랭킹):
| asin       | title_str                                                                                                                                                                                     | domain      |   final_score |
|:-----------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:------------|--------------:|
| B018YY3WTE | Karaoke Machine for Adults and Kids                                                                                                                                                           | instruments |      0.786173 |
| B0871S2S7Q | HIKEP Bluetooth Karaoke Machine with Disco Lights Wireless Microphone, Portable PA System Speaker for Kids Adults for Parties Karaoke                                                       

In [ ]:
############################################################
# Evaluation 
############################################################

import pandas as pd
import numpy as np
import os
from collections import defaultdict
from sklearn.metrics import ndcg_score

# ---------------------------------------------------------
# 1️ 데이터 로드
# ---------------------------------------------------------
BASE_DIR = '../'
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

reviews_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv'))
meta_df = pd.read_pickle(os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl'))

# ---------------------------------------------------------
# 2️ Ground Truth (사용자별 실제 선호 아이템)
# ---------------------------------------------------------
def build_ground_truth(df, threshold=4.0):
    liked = df[df['overall'] >= threshold]
    gt = liked.groupby('reviewerID')['asin'].apply(list).to_dict()
    return gt

ground_truth = build_ground_truth(reviews_df)

# ---------------------------------------------------------
# 3️ 평가지표 함수
# ---------------------------------------------------------
def precision_at_k(pred, gt, k):
    hit = sum(1 for item in pred[:k] if item in gt)
    return hit / k

def recall_at_k(pred, gt):
    if len(gt) == 0:
        return np.nan
    return len(set(pred) & set(gt)) / len(gt)

def f1_at_k(prec, rec):
    if prec + rec == 0:
        return 0
    return 2 * prec * rec / (prec + rec)

def ndcg_at_k(pred, gt, k):
    relevance = [1 if item in gt else 0 for item in pred[:k]]
    ideal = sorted(relevance, reverse=True)
    if sum(ideal) == 0:
        return 0
    return ndcg_score([ideal], [relevance])

def mrr_at_k(pred, gt):
    for rank, item in enumerate(pred, start=1):
        if item in gt:
            return 1 / rank
    return 0

# ---------------------------------------------------------
# 4️ 공통 평가 루프
# ---------------------------------------------------------
def evaluate_model(get_recommendations_func, model_name="MODEL", top_k=10, max_users=200):
    precisions, recalls, f1s, ndcgs, mrrs = [], [], [], [], []
    user_list = list(ground_truth.keys())[:max_users]

    for uid in user_list:
        gt_items = ground_truth.get(uid, [])
        if not gt_items:
            continue

        rec_items = get_recommendations_func(uid)
        if not rec_items:
            continue

        prec = precision_at_k(rec_items, gt_items, top_k)
        rec = recall_at_k(rec_items, gt_items)
        f1  = f1_at_k(prec, rec)
        ndcg = ndcg_at_k(rec_items, gt_items, top_k)
        mrr = mrr_at_k(rec_items, gt_items)

        precisions.append(prec)
        recalls.append(rec)
        f1s.append(f1)
        ndcgs.append(ndcg)
        mrrs.append(mrr)

    print(f"\n📊 {model_name} Evaluation Results (Top-{top_k})")
    print(f"Precision@{top_k}: {np.nanmean(precisions):.4f}")
    print(f"Recall@{top_k}   : {np.nanmean(recalls):.4f}")
    print(f"F1@{top_k}       : {np.nanmean(f1s):.4f}")
    print(f"NDCG@{top_k}     : {np.nanmean(ndcgs):.4f}")
    print(f"MRR@{top_k}      : {np.nanmean(mrrs):.4f}")

# ---------------------------------------------------------
# 5️ 추천 함수 (모델별)
# ---------------------------------------------------------
# Trend Model
def get_trend_recs(user_id, k=10):
    return list(trend_scores.sort_values(ascending=False).head(k).index)

# Content-Based (TF-IDF)
def get_cb_recs(user_id, k=10):
    liked_asin = reviews_df[reviews_df['reviewerID'] == user_id]['asin'].iloc[0]
    scores = get_cb_score(liked_asin).sort_values(ascending=False)
    return list(scores.head(k).index)

# Collaborative Filtering (SVD)
def get_cf_score(user_id):
    asins_music = meta_df.loc[meta_df['domain'] == 'music', 'asin'].unique()
    preds = []
    for iid in asins_music:
        try:
            pred = svd_model.predict(user_id, iid).est
        except:
            pred = 0
        preds.append(pred)
    return pd.Series(preds, index=asins_music)

def get_cf_recs(user_id, k=10):
    scores = get_cf_score(user_id).sort_values(ascending=False)
    return list(scores.head(k).index)

# Hybrid (Re-ranking)
def get_hybrid_recs(user_id, k=10):
    liked_asin = reviews_df[reviews_df['reviewerID'] == user_id]['asin'].iloc[0]
    result = hybrid_recommend(user_id, liked_asin, top_n=k)
    return list(result['asin'].values)

# ---------------------------------------------------------
# 6️ 평가 실행
# ---------------------------------------------------------
evaluate_model(get_trend_recs, "Trend (Popularity)", top_k=10)
evaluate_model(get_cb_recs, "Content-Based (TF-IDF)", top_k=10)
evaluate_model(get_cf_recs, "Collaborative (SVD, Music-only)", top_k=10)
evaluate_model(get_hybrid_recs, "Hybrid (Re-ranking)", top_k=10)



📊 Trend (Popularity) Evaluation Results (Top-10)
Precision@10: 0.0005
Recall@10   : 0.0002
F1@10       : 0.0003
NDCG@10     : 0.0020
MRR@10      : 0.0007

📊 Content-Based (TF-IDF) Evaluation Results (Top-10)
Precision@10: 0.0925
Recall@10   : 0.2087
F1@10       : 0.1123
NDCG@10     : 0.8309
MRR@10      : 0.8425

📊 Collaborative (SVD, Music-only) Evaluation Results (Top-10)
Precision@10: 0.0005
Recall@10   : 0.0005
F1@10       : 0.0005
NDCG@10     : 0.0020
MRR@10      : 0.0007

📊 Hybrid (Re-ranking) Evaluation Results (Top-10)
Precision@10: 0.0050
Recall@10   : 0.0075
F1@10       : 0.0055
NDCG@10     : 0.0119
MRR@10      : 0.0129
